# Extract Video Insights Using Bedrock Data Automation
Amazon Bedrock Data Automation is a GenAI-powered capability of Bedrock that streamlines the development of generative AI applications and automates workflows involving documents, images, audio, and videos. 

By leveraging Bedrock Data Automation, developers can reduce development time and effort, making it easier to build intelligent document processing, media analysis, and other multimodal data-centric automation solutions. 

Bedrock Data Automation offers industry-leading accuracy at lower cost than alternative solutions, along with features such as visual grounding with confidence scores for explainability and built-in hallucination mitigation. 

This ensures trustworthy and accurate insights from unstructured, multi-modal data sources. Bedrock Data Automation is also integrated with Bedrock Knowledge Bases, making it easier for developers to generate meaningful information from their unstructured multi-modal content to provide more relevant responses for retrieval augmented generation (RAG).

In this notebook, we'll leverage Bedrock data automation to process the media content, and ingest the insights as embeddings into Bedrock Knowledge Bases. Through these embeddings, the conversation search agent will be able to answer questions related to the show. 

## Prerequisites
First, let's install all the required libraries for running the notebook end to end.

In [ ]:
# Make sure you download the latest botocore and boto3 libraries.
import shutil
import subprocess
import sys

def ensure_uv_installed():
    if shutil.which("uv") is None:
        print("🔧 'uv' not found. Installing with pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"])
    else:
        print("✅ 'uv' is already installed.")

def uv_install(*packages):
    ensure_uv_installed()
    uv_path = shutil.which("uv")
    print(f"📦 Installing {', '.join(packages)} using uv...")
    subprocess.check_call([uv_path, "pip", "install", *packages])

uv_install("botocore", "boto3")

In [ ]:
!uv pip install -r requirements.txt

Import relevant libraries

In [ ]:
import boto3
import json
import uuid
import utils
import os
import base64
import io
import time
import subprocess
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime
from termcolor import colored
from pprint import pprint
from moviepy.video.io.VideoFileClip import VideoFileClip
from contextlib import redirect_stdout, redirect_stderr
from dotenv import load_dotenv
from IPython.display import clear_output
from IPython.display import display, JSON, HTML, Video
from utils import (add_video_metadata_attributes, 
                   display_chapter_videos, 
                   find_shots, 
                   find_key_frames, 
                   detect_celebrities, 
                   display_search_results, 
                   clip_video, 
                   display_chapter_segment_videos)
from oss_utils import OpenSearchHelper

In [ ]:
import os
os.environ["AWS_PROFILE"] = "your AWS Profile" # use profile or AWS Access Keys
os.environ["AWS_DEFAULT_REGION"] = "AWS region"

In [ ]:
embedding_model_id = "amazon.nova-2-multimodal-embeddings-v1:0"
embedding_dim = 1024
bedrock_runtime_client = boto3.client('bedrock-runtime')

In [ ]:
bedrock_agent_client = boto3.client("bedrock-agent")

bedrock_runtime_agent_client = boto3.client('bedrock-agent-runtime')

bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
rekognition_client = boto3.client('rekognition')
s3_client = boto3.client('s3')
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
region = boto3.Session().region_name
rerank_model_id = "amazon.rerank-v1:0"
rerank_model_arn = f"arn:aws:bedrock:{region}::foundation-model/{rerank_model_id}"
print(f'Current AWS account Id: {account_id}, region name: {region}')

oss_host = "an opensearch collection host URL" # replace this with the output from CDK: OpenSearchServerlessStack.OpenSearchCollectionEndpoint
oss_helper = OpenSearchHelper(host=oss_host.removeprefix("https://"))
video_index_name = "bda-video-level-detail-index"
video_chapter_index_name = "bda-video-chapter-level-detail-index"
video_chapter_audio_segment_index_name = "bda-video-chapter-audio-segment-level-detail-index"
video_chapter_segment_index_name = "bda-video-chapter-segment-level-detail-index"


In the following, we are using an existing S3 bucket for storage throughout the notebook. We recommend creating a new S3 bucket in the same region where you plan to run the notebook. 

In [ ]:
data_bucket = "enter your bucket name here" # You can find the bucket name in the CDK output with the output name: NovaSonicSolutionBackendStack.MediaContentBucketName
data_prefix = "input/bda"
output_prefix = "output"
print('Video data S3 bucket:', data_bucket)

## Create a BDA project with a standard output configuration for all video assets
To start a BDA job, you need a BDA project, which organizes both standard and custom output configurations. This project is reusable, allowing you to apply the same configuration to process multiple videos that share the same settings. For more information about BDA project and the features, please follow this [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-projects.html)

In the code snippet below, we create a BDA project with standard output configurations for video modality. These configurations can be tailored to extract only the specific information you need. In this lab, we will enable the below video outputs:

- Full video summary
- Chapter summaries
- IAB categories on the chapter level
- Full audio transcript
- Text in video with bounding-boxes
- Brand & logo in video with bounding-boxes

For a complete API reference for creating a BDA project, refer to this [document](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-data-automation/client/create_data_automation_project.html).

In [ ]:
response = bda_client.create_data_automation_project(
    projectName=f'bda-project-{str(uuid.uuid4())[0:4]}',
    projectDescription='BDA Conversation Agent sample project',
    projectStage='DEVELOPMENT',
    standardOutputConfiguration={
        'video': {
            'extraction': {
                'category': {
                    'state': 'ENABLED',
                    'types': ['TEXT_DETECTION','TRANSCRIPT','LOGOS'],
                },
                'boundingBox': {
                    'state': 'ENABLED',
                }
            },
            'generativeField': {
                'state': 'ENABLED',
                'types': ['VIDEO_SUMMARY','CHAPTER_SUMMARY','IAB'],
            }
        }
    }
)

The create_data_automation_project API will return the project ARN, which we will use it to invoke the video analysis task.

In [ ]:
video_project_arn = response.get("projectArn")
print("BDA video project ARN:", video_project_arn)


## Start an asynchronous BDA task to extract and analyze a given video.
In this section, we will use a open-source movie Meridian, and extract and analyze it using BDA, applying the configuration defined in the BDA project. We will then review the output to gain a deeper understanding of how BDA performs video extraction and analysis.

### Prepare the sample video
In this example, we use a video content called "Tears of Steel", an open source movie content available to download in this [URL](https://studio.blender.org/projects/tears-of-steel/gallery/?asset=42). Download the movie into a local directory, and provide the local path in the following section to process the video file in BDA. 
  

In [ ]:
sample_video = 'the local mp4 file downloaded from the URL above'
original_video_name = os.path.basename(sample_video)
sample_video_url = f"s3://{data_bucket}/{data_prefix}/{original_video_name}"

To analyze the video using BDA, we need to upload it to an S3 bucket that BDA can access. 

In [ ]:
!aws s3 cp $sample_video $sample_video_url

### Start BDA task
We will now invoke the BDA API to process the uploaded video. You need to provide the BDA project ARN that we created at the beginning of the lab and specify an S3 location where BDA will store the output results.

For a complete API reference for invoke a BDA async task, refer to this [document](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-data-automation-runtime/client/invoke_data_automation_async.html).

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': sample_video_url
    },
    outputConfiguration={
        's3Uri': f's3://{data_bucket}/{output_prefix}'
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': video_project_arn,
        'stage': 'DEVELOPMENT'
    },
    notificationConfiguration={
        'eventBridgeConfiguration': {
            'eventBridgeEnabled': False
        }
    },
    dataAutomationProfileArn=f'arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

The `invoke_data_automation_async` API is asynchronous. It returns an invocation task identifier, `invocationArn`. We can then use another API `get_data_automation_status` to monitor the task's status until it completes.

> In production workloads, an event-driven pattern is recommended. Allow BDA to trigger the next step once the task is complete. This can be achieved by configuring the notificationConfiguration in the invoke task, which will send a notification to a subscribed AWS service, such as a Lambda function. Alternatively, you can set up an S3 trigger on the bucket where BDA will drop the results.

In [ ]:
invocation_arn = response.get("invocationArn")
print("BDA task started:", invocation_arn)

The job submission API returns a BDA task ID, which we can use to monitor the status by calling the `get_data_automation_status` API every 5 seconds until the task is complete.

This video will take ~3 minutes.

In [ ]:
status, status_response = None, None
while status not in ["Success","ServiceError","ClientError"]:
    status_response = bda_runtime_client.get_data_automation_status(
        invocationArn=invocation_arn
    )
    status = status_response.get("status")
    clear_output(wait=True)
    print(f"{datetime.now().strftime('%H:%M:%S')} : BDA video task: {status}")
    time.sleep(5)

output_config = status_response.get("outputConfiguration",{}).get("s3Uri")
print("Output configureation file:", output_config)

## Access the BDA analysis result
The `get_data_automation_status` API returns an S3 URI containing the result configuration, which provides the S3 location where BDA outputs the extraction results. We will then parse this file to retrieve the result path.

In [ ]:
config_data = utils.read_json_on_s3(output_config,s3_client)
pprint(config_data)

As shown above, the BDA output configuration file contains metadata about the BDA result, including the job ID, status, modality, and the S3 location of the actual result JSON. We will now download this result file to verify the output.

<div class=\"alert alert-info\"> 💡 Notice in the JSON output, the element <b>semantic_modality</b> is <b>VIDEO</b>. This is because BDA identifies the content to process is a video content. In addition to Video, BDA also supports other modalities, inluding Documents and Images. You use the same API call across all these modalities without changing the API structure. Please see <a href="https://docs.aws.amazon.com/bedrock/latest/userguide/bda-standard-output.html">here</a> for more detail to learn more.
</div>

In [ ]:
Path("temp-outputs/bda").mkdir(parents=True, exist_ok=True)

result_uri = config_data["output_metadata"][0]["segment_metadata"][0]["standard_output_path"]
result_data = utils.read_json_on_s3(result_uri,s3_client)

# Because the file is large, we are going to save this file so we could view the content using the built-in JSON file explorer in your IDE or jupyterlab.
with open("temp-outputs/bda/output.json", "w") as f:
    f.write(json.dumps(result_data))

## Review the result
The BDA video analysis result contains a detailed breakdown of information, organized by video and chapter level.
> A video chapter is a sequence of shots that form a coherent unit of action or narrative within the video. This feature breaks down the video into meaningful segments based on visual and audible cues, provides timestamps for those segments, and summarizes each. 

### Full video summary

Let's take a look at the video level summary - it distills the key themes, events, and information presented throughout the video into a concise summary. 

In [ ]:
video_summary = result_data["video"]["summary"]
print(colored(video_summary, 'green'))

# Analyze video level insights
The BDA job produced a lot of valuable insights like the one shown above for the given media content. The insights include video level summariziation, chapters, transcription, audio segments, texts identified in the video frames. In the following sections, we'll leverage all the information to build a rich set of metadata to power the search experience for the conversation media agent.


### Full video transcription
BDA outputs the full transcription with speaker diarization. In the following, let's see the full transcription extracted by the BDA job.

In [ ]:
print(result_data["video"]["transcript"]["representation"]["text"])

# Generate Video Level Insights
In the last step, we saw the insights generated by BDA at the video level. In the following steps, we'll capture the data and transform them into information that can be used for semantic search.

First, we'll define the variables to be used for storing the relevant contents to S3 bucket so that they could be converted into an embedding and stored in a vector database.

In [ ]:
# Extract the original video name and the extension.
# task_id = invocation_arn.split("/")[-1]
arr = result_data["metadata"]["s3_key"].split('/')[-1].split('.')
video_file_name = arr[0]
video_format = arr[1]

In [ ]:
from typing import Any, Dict, Generator, List, Optional

def invoke_model(input_body: Dict[str, Any] = {}) -> Dict[str, Any]:
        body = json.dumps(input_body)

        try:
            response = bedrock_runtime_client.invoke_model(
                body=body,
                modelId=embedding_model_id,
                accept="application/json",
                contentType="application/json",
            )

            response_metadata = response.get("ResponseMetadata", {})
            response_body = json.loads(response.get("body").read())
            return response_body
        except Exception as e:
            raise e

def embedding_func(text: str, input_type: str = "GENERIC_RETRIEVAL") -> List[float]:
    """Call out to Bedrock embedding endpoint with a single text."""
    # replace newlines, which can negatively affect performance.
    text = text.replace(os.linesep, " ")

    # Create embedding
    request_body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": input_type,
            "embeddingDimension": embedding_dim,
            "text": {"truncationMode": "END", "value": text},
        },
    }

    response = invoke_model(input_body=request_body)
    embedding = response["embeddings"][0]["embedding"]
    return embedding

In [ ]:
from itertools import batched

def bulk_ingest_data(data_collection, index_name):
    data_bulk_collection = []
    for data in data_collection:
        data_bulk = {}
        data_bulk["_index"] = index_name
        data_bulk["_source"] = data
        data_bulk_collection.append(data_bulk)

    batch_size = 20
    oss_bulk_batches = [list(batch) for batch in batched(data_bulk_collection, batch_size)]
    for oss_bulk_batch in oss_bulk_batches:
        results = oss_helper.bulk_index(oss_bulk_batch)

In [ ]:
# Create a text file that contains the video level summary, then upload to S3 bucket. 
summary = result_data["video"]["summary"]
video_level_embedding = embedding_func(summary)
video_metadata = {
    "embedding" : video_level_embedding,
    "summary" : summary,
    "video_name": original_video_name,
    "start_timestamp_millis" : 0,
    "end_timestamp_millis" : result_data["metadata"]["duration_millis"],
    "src_media_s3_location": sample_video_url
}

video_level_doc= []
video_level_doc.append(video_metadata)

In [ ]:
bulk_ingest_data(video_level_doc, video_index_name)

# Chapter Level Details
In addition to the video level insights, BDA also generates a chapter-level insights from the video. Each chapter includes a chapter summary and other useful metadata, including the start and end times of each chapter, the audio segments, and shots within the chapters. We'll leverage these information to build a rich semantic search engine to power the search for the video.

Specifically, in the following section, we'll extract the chapter level summary, texts detected on the video content and the chapter level transcription.

In [ ]:
chapter_details = []
for idx, s in enumerate(result_data["chapters"]):
    txts, chapter_transcriptions = [],[]
    if 'transcript' in s:
        scene_transcription = s['transcript']['representation']['text']
        chapter_transcriptions.append(scene_transcription)

    
    # if txts:
    #     summary += f"\nText lines detected in the chapter: {','.join(txts)}"
    summary = f"{s['summary']}"
    if chapter_transcriptions:
        summary += f"\n{','.join(chapter_transcriptions)}"
    chapter_detail = {}
    chapter_detail["embedding"] = embedding_func(summary)
    chapter_detail["chapter"] = idx + 1
    chapter_detail["summary"]=summary
    chapter_detail["chapter_transcription"] = chapter_transcriptions
    chapter_detail["start_timestamp_millis"]=s['start_timestamp_millis'] 
    chapter_detail['end_timestamp_millis']=s["end_timestamp_millis"]
    chapter_detail["video_name"] = original_video_name
    chapter_detail["src_media_s3_location"] = sample_video_url
    chapter_details.append(chapter_detail)

In [ ]:
bulk_ingest_data(chapter_details, video_chapter_index_name)

The shot level details are useful for contents that are visual dominant. Since we are working with news contents, shot level information might be too granular to be included. We are showing the details for your reference. Feel free to experiment with shot level details. In our notebook, we'll not be using shot level details for our video knowledge source. 

# Create Chapter Segment Insights
In addition to full transcript at the chapter level, BDA also extracts the audio segments within the chapters to provide a finer granularity from the audio. The segments are typically broken down by sentences, or long pauses within the speech in the chapter. Furthermore, these audio segments contain the timecodes so that we know exactly when particular sentences are spoken, or a term was brought up. In the context of news content, these audio segments could provide another level of detail beyond the chapter level insights, and enable users who are looking for either a broader topic, or a more detailed level insights. 

In our notebook, we'll capture the chapter segment audio along with the chapter level insights so that they could be made available via semantic search.


In [ ]:
import math 
def find_key_frames_from_shot(shot, max_frames=5):
    shot_frame_count = shot["end_frame_index"] - shot["start_frame_index"] + 1
    frame_distance = max(math.ceil(shot_frame_count/max_frames), 1)
    segment_key_frames = []
    for key_frame_idx in range(shot["start_frame_index"], shot["end_frame_index"], frame_distance):
        segment_key_frame = {}
        segment_key_frame["index"] = key_frame_idx
        segment_key_frames.append(segment_key_frame)
    return segment_key_frames

In [ ]:
chapter_segments = []
shots_data = result_data["shots"]
with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull), redirect_stderr(fnull):
        with VideoFileClip(sample_video) as video_clip:
            fps = video_clip.fps
            for chapter in result_data["chapters"]: 
                shot_indices = chapter["shot_indices"]
                chapter_segment_arr = []
                for idx, shot_index in enumerate(shot_indices):
                    chapter_segment_dict = {}
                    chapter_segment_dict["chapter"] = chapter["chapter_index"] + 1
                    chapter_segment_dict["video_name"] = original_video_name
                    chapter_segment_dict["src_media_s3_location"] = sample_video_url
                    chapter_segment_dict["chapter_segment"] = shot_index + 1

                    chapter_segment_dict["start_timestamp_millis"] =  shots_data[shot_index]["start_timestamp_millis"]
                    chapter_segment_dict["end_timestamp_millis"] =  shots_data[shot_index]["end_timestamp_millis"]
                    chapter_segment_key_frames = find_key_frames_from_shot(shots_data[shot_index])
                    image_frames = []
                    for key_frame in chapter_segment_key_frames:
                        time = key_frame['index'] / fps
                        shot_frame = video_clip.get_frame(time) 
                        shot_frame_bytes = Image.fromarray(shot_frame)
                        image_frames.append(shot_frame_bytes)
                    chapter_segment_dict["key_frame_images"] = image_frames
                    chapter_segment_arr.append(chapter_segment_dict)
                chapter_segments.append(chapter_segment_arr)

In [ ]:
chapter_audio_segments = []
for chapter in result_data["chapters"]: 
    audio_segments = chapter["audio_segments"]
    audio_segment_arr = []
    for idx, audio_segment in enumerate(audio_segments):
        audio_segment_dict = {}
        audio_segment_dict["chapter"] = chapter["chapter_index"] + 1
        audio_segment_dict["video_name"] = original_video_name
        audio_segment_dict["src_media_s3_location"] = sample_video_url
        audio_segment_dict["chapter_segment"] = idx + 1
        audio_segment_dict["start_timestamp_millis"] =  audio_segment["start_timestamp_millis"]
        audio_segment_dict["end_timestamp_millis"] =  audio_segment["end_timestamp_millis"]
        audio_segment_dict["text"] = audio_segment["text"]
        audio_segment_arr.append(audio_segment_dict)
    chapter_audio_segments.append(audio_segment_arr)

In [ ]:
indexable_chapter_audio_segments = []
for chapter_audio_segment in chapter_audio_segments:
    for audio_segment in chapter_audio_segment:
        indexable_chapter_audio_segments.append(audio_segment)

In [ ]:
bulk_ingest_data(indexable_chapter_audio_segments, video_chapter_audio_segment_index_name)

# Detect Celebrities within the Video
Celebrities appearances are common in media contents. At this writing, celebrity detection feature is on the BDA roadmap. For now, we'll use Amazon Rekognition API to perform celebrity detection from the key frames within the chapter segments. We'll incorporate the detected celebrities into the metadata to help provide additional  insights about the news content. The following section takes the key frames from the chapter segment, and perform celebrity detection on selected images.

In [ ]:
import cv2
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [ ]:
import numpy as np
from botocore.exceptions import ClientError

def detect_celebrities(rekognition_client, frame, chapter_idx=0, shot_idx=0, frame_idx=0):
    celebrities = []
    try:
        img_byte_arr = io.BytesIO()
        frame.save(img_byte_arr, format='JPEG')
        img_bytes = img_byte_arr.getvalue()
        nparr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(img, scaleFactor=1.01, minNeighbors=100, minSize=(50, 50))
        if len(faces) > 0:
            # for (x, y, w, h) in faces:
            #     cv2.rectangle(img, (x, y), (x+w, y+h), (255, 255, 0), 2)
            # cv2.imwrite(f'faces/output-{chapter_idx}-{shot_idx}-{frame_idx}.jpg', img)
            response = rekognition_client.recognize_celebrities(
                Image={'Bytes': img_bytes}
            )
            min_confidence = 99.0 # change this value if the accuracy is low.
            for celebrity in response.get('CelebrityFaces', []):
                if celebrity.get('MatchConfidence', 0.0) >= min_confidence:
                    celebrities.append(celebrity['Name'])
        return celebrities

    except ClientError as e:
        print(e)
        pass

In [ ]:
for chapter_segment in chapter_segments:
    chapter_idx = int(chapter_segment[0]["chapter"])    
    for shot_idx, segment_detail in enumerate(chapter_segment):
        key_frame_celebrities = []
        for frame_idx, key_frame_image in enumerate(segment_detail["key_frame_images"]):
            detected_celebrities = detect_celebrities(rekognition_client, key_frame_image, chapter_idx, shot_idx, frame_idx)
            key_frame_celebrities.extend(detected_celebrities)
        key_frame_celebrities = list(set(key_frame_celebrities))
        segment_detail["celebrities"] = key_frame_celebrities


In [ ]:
indexable_chapter_segments = []
for chapter_segment in chapter_segments:
    for segment in chapter_segment:
        segment_data = {}
        segment_data["video_name"] = segment["video_name"]
        segment_data["start_timestamp_millis"] = segment["start_timestamp_millis"]
        segment_data["end_timestamp_millis"] = segment["end_timestamp_millis"]
        segment_data["src_media_s3_location"] = segment["src_media_s3_location"]
        segment_data["chapter"] = segment["chapter"]
        segment_data["chapter_segment"] = segment["chapter_segment"]
        segment_data["celebrities"] = segment["celebrities"]
        indexable_chapter_segments.append(segment_data)

In [ ]:
bulk_ingest_data(indexable_chapter_segments, video_chapter_segment_index_name)